# 09.02 开放寻址批量查询实验

本实验从一个可验证的 CPU 开放寻址表出发，逐步完成数据生成、探测长度统计、多核 Tiling、Ascend C Kernel、C++ Host、CMake 构建、NPU 运行和逐元素验证。实验重点是理解装填因子与线性探测访存路径的关系，而不是仅观察一次运行时间。

## 小节概述

### 前置要求

- 能够阅读 Python、C/C++ 和 Bash（命令行脚本）代码。
- 理解哈希冲突、线性探测和数组下标回绕。
- 能够按顺序运行 Notebook Code Cell（代码单元格）。

### 本节目标

本节使用 SoA（Structure of Arrays，结构分离数组）保存哈希表，以 Query（查询）作为批处理输入，并通过 Tiling（数据切分）、Host（主机侧程序）、Kernel（NPU 核函数）、UB（Unified Buffer，统一缓冲区）、GM（Global Memory，全局内存）和 Golden（参考结果）完成端到端验证。

1. 构建由 `table_keys/table_values/states` 组成的 SoA 哈希表。
2. 正确实现 EMPTY（空槽）、FULL（有效槽）、Tombstone（墓碑状态）查询语义。
3. 统计平均、P95、P99 和最大探测长度。
4. 完成 `QUERY_TILE=128` 的多核 Ascend C 批量查询。
5. 使用 CPU Golden 精确验证 value、hit 和 probe count。

## 0. 创建 Notebook 工作区

为避免 CMake 解析中文绝对路径时出现转义问题，实际工程默认生成到课程根目录的纯英文路径 `work/09.02_open_addressing_lookup`。也可以在启动 Notebook 前设置 `OPEN_ADDRESS_WORK_DIR` 指向其他纯英文目录。

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import numpy as np

# 本单元只定位课程根目录并创建工作区；后续 %%writefile 都依赖 WRITE_ROOT。
cwd = Path.cwd().resolve()
chapter_name = "09_load_factor_and_linear_probing"
if (cwd / chapter_name).is_dir():
    COURSE_DIR = cwd
elif cwd.name == chapter_name:
    COURSE_DIR = cwd.parent
else:
    raise FileNotFoundError("请从课程根目录或 09 章节目录启动 Notebook")

override = os.environ.get("OPEN_ADDRESS_WORK_DIR")
WORK_DIR = Path(override).expanduser().resolve() if override else (
    COURSE_DIR / "work" / "09.02_open_addressing_lookup"
)
if any(ord(ch) > 127 for ch in str(WORK_DIR)):
    raise ValueError("编译工作区必须使用纯英文路径，请设置 OPEN_ADDRESS_WORK_DIR")

for subdir in ("scripts", "op_kernel", "op_host", "input", "output"):
    (WORK_DIR / subdir).mkdir(parents=True, exist_ok=True)
WRITE_ROOT = WORK_DIR.as_posix()
print("课程根目录：", COURSE_DIR)
print("工程写入目录：", WORK_DIR)

## 1. 检查 CANNLab 开发环境

In [ ]:
# CPU 教学部分不依赖 NPU；只有检测到 CANN 环境与设备后才执行构建和上板单元。
ascend_home = os.environ.get("ASCEND_HOME_PATH")
npu_smi = shutil.which("npu-smi")
NPU_READY = bool(ascend_home and Path(ascend_home, "set_env.sh").exists() and npu_smi)
print("ASCEND_HOME_PATH:", ascend_home or "未设置")
print("npu-smi:", npu_smi or "未找到")
print("NPU_READY:", NPU_READY)
if not NPU_READY:
    print("当前可继续运行 CPU 构表、统计与练习；NPU 单元请在 CANNLab 910B3 环境执行。")

## 2. 明确算子接口与状态语义

<table align="left" style="width: 90%; max-width: 1200px; margin: 0 auto 0 0 !important; text-align: left;">
  <thead><tr>
    <th style="text-align: left;">参数</th><th style="text-align: left;">类型与形状</th><th style="text-align: left;">读写</th><th style="text-align: left;">作用</th>
  </tr></thead>
  <tbody>
    <tr><td style="text-align: left;"><code>table_keys</code></td><td style="text-align: left;"><code>int32[M]</code></td><td style="text-align: left;">只读</td><td style="text-align: left;">槽位中的有效 key</td></tr>
    <tr><td style="text-align: left;"><code>table_values</code></td><td style="text-align: left;"><code>int32[M]</code></td><td style="text-align: left;">只读</td><td style="text-align: left;">命中后才读取的 value</td></tr>
    <tr><td style="text-align: left;"><code>states</code></td><td style="text-align: left;"><code>int32[M]</code></td><td style="text-align: left;">只读</td><td style="text-align: left;">0=EMPTY，1=FULL，2=TOMBSTONE</td></tr>
    <tr><td style="text-align: left;"><code>query_keys</code></td><td style="text-align: left;"><code>int32[Q]</code></td><td style="text-align: left;">只读</td><td style="text-align: left;">按连续 Tile 搬入的批量查询</td></tr>
    <tr><td style="text-align: left;"><code>out_values</code></td><td style="text-align: left;"><code>int32[Q]</code></td><td style="text-align: left;">只写</td><td style="text-align: left;">命中 value，未命中为 0</td></tr>
    <tr><td style="text-align: left;"><code>hit_flags</code></td><td style="text-align: left;"><code>int32[Q]</code></td><td style="text-align: left;">只写</td><td style="text-align: left;">命中为 1，未命中为 0</td></tr>
    <tr><td style="text-align: left;"><code>probe_counts</code></td><td style="text-align: left;"><code>int32[Q]</code></td><td style="text-align: left;">只写</td><td style="text-align: left;">本次查询实际检查的槽位数</td></tr>
  </tbody>
</table>
<div style="clear: both;"></div>

## 3. 编写 CPU 构表、查询与统计工具

CPU 工具是数据生成器、Golden 和边界验证的共同依据。哈希函数、回绕公式和 probe count 定义必须与 Kernel 完全一致。

In [ ]:
%%writefile $WRITE_ROOT/scripts/open_addressing_utils.py
from __future__ import annotations

from dataclasses import dataclass
import numpy as np

EMPTY, FULL, TOMBSTONE = 0, 1, 2
U32_MASK = 0xFFFFFFFF


def is_power_of_two(value):
    return value > 0 and (value & (value - 1)) == 0


def hash32(key):
    # 哈希体现：转为 uint32 后做乘法与高低位混合，三端必须逐步一致。
    value = int(key) & U32_MASK
    value = (value * 0x9E3779B1) & U32_MASK
    value ^= value >> 16
    return value & U32_MASK


def validate_arrays(table_keys, table_values, states, max_probe):
    if not (len(table_keys) == len(table_values) == len(states)):
        raise ValueError("table arrays must have equal length")
    table_size = len(states)
    if not is_power_of_two(table_size):
        raise ValueError("table_size must be a positive power of two")
    if not 1 <= max_probe <= table_size:
        raise ValueError("max_probe must be in [1, table_size]")
    if not np.all(np.isin(states, [EMPTY, FULL, TOMBSTONE])):
        raise ValueError("state must be EMPTY, FULL or TOMBSTONE")
    seen = set()
    mask = table_size - 1
    for slot in np.flatnonzero(states == FULL):
        key = int(table_keys[slot])
        if key in seen:
            raise ValueError("duplicate FULL key")
        seen.add(key)
        home = hash32(key) & mask
        distance = (int(slot) - home) & mask
        if distance >= max_probe:
            raise ValueError("a FULL key is outside max_probe")
        for step in range(distance):
            if int(states[(home + step) & mask]) == EMPTY:
                raise ValueError("an EMPTY slot breaks the probe chain")


@dataclass
class LookupResult:
    value: int
    hit: int
    probes: int


class LinearProbeTable:
    def __init__(self, table_size):
        if not is_power_of_two(table_size):
            raise ValueError("table_size must be a positive power of two")
        self.keys = np.zeros(table_size, dtype=np.int32)
        self.values = np.zeros(table_size, dtype=np.int32)
        self.states = np.zeros(table_size, dtype=np.int32)
        self.size = 0

    @property
    def table_size(self):
        return int(self.states.size)

    @property
    def load_factor(self):
        return self.size / self.table_size

    def insert(self, key, value):
        # 线性探测体现：冲突后只向后移动一个槽，并在表尾按 mask 回绕。
        mask = self.table_size - 1
        home = hash32(key) & mask
        first_tombstone = None
        for step in range(self.table_size):
            slot = (home + step) & mask
            state = int(self.states[slot])
            if state == FULL and int(self.keys[slot]) == int(key):
                self.values[slot] = np.int32(value)
                return step + 1
            if state == TOMBSTONE and first_tombstone is None:
                first_tombstone = slot
            if state == EMPTY:
                target = first_tombstone if first_tombstone is not None else slot
                self.keys[target] = np.int32(key)
                self.values[target] = np.int32(value)
                self.states[target] = FULL
                self.size += 1
                return ((target - home) & mask) + 1
        if first_tombstone is not None:
            self.keys[first_tombstone] = np.int32(key)
            self.values[first_tombstone] = np.int32(value)
            self.states[first_tombstone] = FULL
            self.size += 1
            return ((first_tombstone - home) & mask) + 1
        raise RuntimeError("hash table is full")

    def delete(self, key):
        result, slot = self._lookup_with_slot(key, self.table_size)
        if not result.hit:
            return False
        # Tombstone 体现：删除只改变状态，不把槽恢复为 EMPTY。
        self.states[slot] = TOMBSTONE
        self.size -= 1
        return True

    def _lookup_with_slot(self, key, max_probe):
        mask = self.table_size - 1
        home = hash32(key) & mask
        for step in range(max_probe):
            slot = (home + step) & mask
            state = int(self.states[slot])
            probes = step + 1
            if state == EMPTY:
                return LookupResult(0, 0, probes), slot
            if state == FULL and int(self.keys[slot]) == int(key):
                return LookupResult(int(self.values[slot]), 1, probes), slot
            # TOMBSTONE 与不匹配的 FULL 都必须继续探测。
        return LookupResult(0, 0, max_probe), -1

    def lookup(self, key, max_probe):
        return self._lookup_with_slot(key, max_probe)[0]


def batch_lookup(table, queries, max_probe):
    values, hits, probes = [], [], []
    for key in np.asarray(queries, dtype=np.int32):
        result = table.lookup(int(key), max_probe)
        values.append(result.value)
        hits.append(result.hit)
        probes.append(result.probes)
    return (
        np.asarray(values, dtype=np.int32),
        np.asarray(hits, dtype=np.int32),
        np.asarray(probes, dtype=np.int32),
    )


def probe_statistics(probe_counts):
    values = np.asarray(probe_counts, dtype=np.int64)
    if values.size == 0 or np.any(values <= 0):
        raise ValueError("probe counts must be nonempty and positive")
    return {
        "mean": float(values.mean()),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": int(values.max()),
    }


def keys_for_home(home, count, table_size, start=1):
    # 集中碰撞案例：主动筛选初始槽相同的 key，便于观察连续聚集。
    result, key = [], start
    while len(result) < count:
        if (hash32(key) & (table_size - 1)) == home:
            result.append(key)
        key += 1
    return result

### 3.1 编写测试数据生成器

In [ ]:
%%writefile $WRITE_ROOT/scripts/gen_data.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
import numpy as np

from open_addressing_utils import (
    FULL, LinearProbeTable, batch_lookup, keys_for_home,
    probe_statistics, validate_arrays,
)


def random_records(table_size, load_factor, seed):
    rng = np.random.default_rng(seed)
    count = int(table_size * load_factor)
    keys = rng.choice(np.arange(-200000, 200000, dtype=np.int32), size=count, replace=False)
    values = (keys.astype(np.int64) * 17 + 3).astype(np.int32)
    return list(zip(keys.tolist(), values.tolist()))


def make_case(name):
    rng = np.random.default_rng(202609)
    if name == "empty":
        table, queries = LinearProbeTable(64), np.arange(32, dtype=np.int32)
    elif name == "basic":
        table = LinearProbeTable(64)
        records = [(7, 70), (19, 0), (31, -310), (43, 430)]
        for item in records:
            table.insert(*item)
        queries = np.asarray([7, 19, 31, 43, 99, -5], dtype=np.int32)
    elif name == "cluster":
        table = LinearProbeTable(64)
        collision_keys = keys_for_home(5, 12, 64)
        for key in collision_keys:
            table.insert(key, key * 10)
        queries = np.asarray(collision_keys + [99991, 99992], dtype=np.int32)
    elif name == "tombstone":
        table = LinearProbeTable(64)
        collision_keys = keys_for_home(9, 5, 64)
        for key in collision_keys:
            table.insert(key, key + 100)
        table.delete(collision_keys[1])
        queries = np.asarray(collision_keys + [123456], dtype=np.int32)
    elif name == "tail130":
        table = LinearProbeTable(256)
        records = random_records(256, 0.50, 11)
        for item in records:
            table.insert(*item)
        hit_keys = [key for key, _ in records]
        queries = np.asarray((hit_keys + list(range(500000, 500130)))[:130], dtype=np.int32)
    elif name in ("load25", "load50", "load75"):
        factor = {"load25": 0.25, "load50": 0.50, "load75": 0.75}[name]
        table = LinearProbeTable(256)
        records = random_records(256, factor, {0.25: 25, 0.50: 50, 0.75: 75}[factor])
        for item in records:
            table.insert(*item)
        hit_keys = [key for key, _ in records]
        misses = rng.choice(np.arange(500000, 900000, dtype=np.int32), size=512-len(hit_keys), replace=False)
        queries = np.asarray(hit_keys + misses.tolist(), dtype=np.int32)
        rng.shuffle(queries)
    elif name == "negative_zero":
        table = LinearProbeTable(64)
        for item in [(-1, 0), (-2147483648, 9), (0, 0), (2147483647, -1)]:
            table.insert(*item)
        queries = np.asarray([-1, -2147483648, 0, 2147483647, 123], dtype=np.int32)
    else:
        raise ValueError(f"unknown case: {name}")

    # max_probe 由实际构表结果确定并留出 1 个槽余量，最大不超过表长。
    mask = table.table_size - 1
    distances = []
    for slot in np.flatnonzero(table.states == FULL):
        home = __import__("open_addressing_utils").hash32(int(table.keys[slot])) & mask
        distances.append(((int(slot) - home) & mask) + 1)
    max_probe = min(table.table_size, max(8, max(distances, default=1) + 1))
    validate_arrays(table.keys, table.values, table.states, max_probe)
    return table, queries, max_probe


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--case", default="basic")
    args = parser.parse_args()
    table, queries, max_probe = make_case(args.case)
    values, hits, probes = batch_lookup(table, queries, max_probe)
    Path("input").mkdir(exist_ok=True)
    Path("output").mkdir(exist_ok=True)
    table.keys.tofile("input/table_keys.bin")
    table.values.tofile("input/table_values.bin")
    table.states.tofile("input/states.bin")
    queries.tofile("input/query_keys.bin")
    values.tofile("output/golden_values.bin")
    hits.tofile("output/golden_hits.bin")
    probes.tofile("output/golden_probes.bin")
    meta = {
        "case": args.case, "table_size": table.table_size,
        "query_count": int(queries.size), "max_probe": int(max_probe),
        "item_count": int(table.size), "load_factor": table.load_factor,
        "probe_statistics": probe_statistics(probes),
        "table_bytes": int(table.table_size * 3 * np.dtype(np.int32).itemsize),
    }
    Path("meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(meta, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
# 先在 CPU 上生成三个装填因子案例，观察算法探测统计，不依赖 NPU。
cpu_stats = []
for case_name in ("load25", "load50", "load75"):
    subprocess.run(
        [sys.executable, str(WORK_DIR / "scripts/gen_data.py"), "--case", case_name],
        cwd=WORK_DIR, check=True, stdout=subprocess.DEVNULL,
    )
    import json
    meta = json.loads((WORK_DIR / "meta.json").read_text(encoding="utf-8"))
    cpu_stats.append((case_name, meta["load_factor"], meta["table_bytes"], meta["probe_statistics"]))
for name, alpha, table_bytes, stats in cpu_stats:
    print(f"{name}: α={alpha:.2f}, table={table_bytes}B, mean={stats['mean']:.3f}, "
          f"P99={stats['p99']:.3f}, max={stats['max']}")

## 4. 设计多核 Tiling 与 UB 预算

每个 Core 处理一个连续 query 区间，`queriesPerCore=ceil(Q/blockNum)`。每次搬入 `QUERY_TILE=128` 个 `int32` query，并为 value、hit、probe 各准备同样大小的输出 Tile。双缓冲主要数据量为 `4 × 2 × 128 × 4B = 4096B`，远小于 910B3 的 UB 容量。表数组不整体搬入 UB，因为探测槽位由 query 在运行时决定。

In [ ]:
QUERY_TILE = 128
tile_bytes = QUERY_TILE * np.dtype(np.int32).itemsize
ub_bytes = 4 * 2 * tile_bytes
print(f"单个数组 Tile: {tile_bytes}B")
print(f"query + 3 outputs 双缓冲: {ub_bytes}B")

## 5. 编写共享 Tiling 结构

In [ ]:
%%writefile $WRITE_ROOT/op_kernel/open_addressing_lookup_tiling.h
#pragma once
#include <cstdint>

constexpr uint32_t QUERY_TILE = 128;

// Host 与 Kernel 共享这些字段，所有计数均以 int32 元素个数为单位。
struct alignas(8) OpenAddressingLookupTilingData {
    uint32_t tableSize;
    uint32_t queryCount;
    uint32_t maxProbe;
    uint32_t blockNum;
    uint32_t queriesPerCore;
    uint32_t reserved;
};

## 6. 编写 Ascend C Kernel

Query 和三个输出通过 UB 队列连续搬运；随机槽位访问只发生在表数组。Kernel 先读 state，再按需读 key，只有确认命中后才读 value。`probe_counts` 对 `EMPTY` 槽也计一次访问。

In [ ]:
%%writefile $WRITE_ROOT/op_kernel/open_addressing_lookup_kernel.cpp
#include "kernel_operator.h"
#include "open_addressing_lookup_tiling.h"

constexpr int32_t STATE_EMPTY = 0;
constexpr int32_t STATE_FULL = 1;
constexpr int32_t STATE_TOMBSTONE = 2;

class KernelOpenAddressingLookup {
public:
    __aicore__ inline KernelOpenAddressingLookup(AscendC::TPipe *pipe) : pipe_(pipe) {}

    __aicore__ inline void Init(
        GM_ADDR tableKeys, GM_ADDR tableValues, GM_ADDR states, GM_ADDR queryKeys,
        GM_ADDR outValues, GM_ADDR hitFlags, GM_ADDR probeCounts,
        const __gm__ OpenAddressingLookupTilingData *tiling)
    {
        tiling_ = tiling;
        const uint32_t blockIdx = AscendC::GetBlockIdx();
        startQuery_ = blockIdx * tiling_->queriesPerCore;
        const uint32_t remaining = startQuery_ < tiling_->queryCount
            ? tiling_->queryCount - startQuery_ : 0;
        localQueryCount_ = remaining < tiling_->queriesPerCore
            ? remaining : tiling_->queriesPerCore;

        // SoA 表数组保持在 GM 中；每个逻辑槽在三个数组中使用相同下标。
        tableKeysGm_.SetGlobalBuffer((__gm__ int32_t *)tableKeys, tiling_->tableSize);
        tableValuesGm_.SetGlobalBuffer((__gm__ int32_t *)tableValues, tiling_->tableSize);
        statesGm_.SetGlobalBuffer((__gm__ int32_t *)states, tiling_->tableSize);
        queryKeysGm_.SetGlobalBuffer((__gm__ int32_t *)queryKeys, tiling_->queryCount);
        outValuesGm_.SetGlobalBuffer((__gm__ int32_t *)outValues, tiling_->queryCount);
        hitFlagsGm_.SetGlobalBuffer((__gm__ int32_t *)hitFlags, tiling_->queryCount);
        probeCountsGm_.SetGlobalBuffer((__gm__ int32_t *)probeCounts, tiling_->queryCount);

        // query 和三个输出均为 int32 连续 Tile，双缓冲总计约 4KB。
        pipe_->InitBuffer(queryQueue_, 2, QUERY_TILE * sizeof(int32_t));
        pipe_->InitBuffer(valueQueue_, 2, QUERY_TILE * sizeof(int32_t));
        pipe_->InitBuffer(hitQueue_, 2, QUERY_TILE * sizeof(int32_t));
        pipe_->InitBuffer(probeQueue_, 2, QUERY_TILE * sizeof(int32_t));
    }

    __aicore__ inline void Process()
    {
        for (uint32_t base = 0; base < localQueryCount_; base += QUERY_TILE) {
            const uint32_t remain = localQueryCount_ - base;
            const uint32_t validLen = remain < QUERY_TILE ? remain : QUERY_TILE;
            CopyIn(base, validLen);
            Compute(validLen);
            CopyOut(base, validLen);
        }
    }

private:
    __aicore__ inline uint32_t Hash32(int32_t key)
    {
        // 与 Python/C++ Host 一致：uint32 乘法自然截断，再混合高低 16 位。
        uint32_t value = static_cast<uint32_t>(key);
        value *= 0x9E3779B1U;
        value ^= value >> 16;
        return value;
    }

    __aicore__ inline void CopyIn(uint32_t base, uint32_t validLen)
    {
        AscendC::LocalTensor<int32_t> queryLocal = queryQueue_.AllocTensor<int32_t>();
        AscendC::DataCopyExtParams params{
            1, static_cast<uint32_t>(validLen * sizeof(int32_t)), 0, 0, 0};
        AscendC::DataCopyPadExtParams<int32_t> pad{false, 0, 0, 0};
        AscendC::DataCopyPad(
            queryLocal, queryKeysGm_[startQuery_ + base], params, pad);
        queryQueue_.EnQue(queryLocal);
    }

    __aicore__ inline void Compute(uint32_t validLen)
    {
        AscendC::LocalTensor<int32_t> queryLocal = queryQueue_.DeQue<int32_t>();
        AscendC::LocalTensor<int32_t> valueLocal = valueQueue_.AllocTensor<int32_t>();
        AscendC::LocalTensor<int32_t> hitLocal = hitQueue_.AllocTensor<int32_t>();
        AscendC::LocalTensor<int32_t> probeLocal = probeQueue_.AllocTensor<int32_t>();
        const uint32_t mask = tiling_->tableSize - 1;

        for (uint32_t i = 0; i < validLen; ++i) {
            const int32_t key = queryLocal.GetValue(i);
            const uint32_t home = Hash32(key) & mask;
            int32_t value = 0;
            int32_t hit = 0;
            int32_t probes = 0;

            for (uint32_t step = 0; step < tiling_->maxProbe; ++step) {
                const uint32_t slot = (home + step) & mask;
                const int32_t state = statesGm_.GetValue(slot);
                probes = static_cast<int32_t>(step + 1);
                if (state == STATE_EMPTY) {
                    break;
                }
                if (state == STATE_FULL && tableKeysGm_.GetValue(slot) == key) {
                    // SoA 按需读取体现：仅在 state/key 已确认命中后读取 value。
                    value = tableValuesGm_.GetValue(slot);
                    hit = 1;
                    break;
                }
                // TOMBSTONE 与 key 不匹配的 FULL 都继续线性探测。
            }
            valueLocal.SetValue(i, value);
            hitLocal.SetValue(i, hit);
            probeLocal.SetValue(i, probes);
        }
        valueQueue_.EnQue(valueLocal);
        hitQueue_.EnQue(hitLocal);
        probeQueue_.EnQue(probeLocal);
        queryQueue_.FreeTensor(queryLocal);
    }

    __aicore__ inline void CopyOut(uint32_t base, uint32_t validLen)
    {
        AscendC::LocalTensor<int32_t> valueLocal = valueQueue_.DeQue<int32_t>();
        AscendC::LocalTensor<int32_t> hitLocal = hitQueue_.DeQue<int32_t>();
        AscendC::LocalTensor<int32_t> probeLocal = probeQueue_.DeQue<int32_t>();
        AscendC::DataCopyExtParams params{
            1, static_cast<uint32_t>(validLen * sizeof(int32_t)), 0, 0, 0};
        const uint32_t offset = startQuery_ + base;
        AscendC::DataCopyPad(outValuesGm_[offset], valueLocal, params);
        AscendC::DataCopyPad(hitFlagsGm_[offset], hitLocal, params);
        AscendC::DataCopyPad(probeCountsGm_[offset], probeLocal, params);
        valueQueue_.FreeTensor(valueLocal);
        hitQueue_.FreeTensor(hitLocal);
        probeQueue_.FreeTensor(probeLocal);
    }

    AscendC::TPipe *pipe_;
    const __gm__ OpenAddressingLookupTilingData *tiling_;
    AscendC::GlobalTensor<int32_t> tableKeysGm_, tableValuesGm_, statesGm_;
    AscendC::GlobalTensor<int32_t> queryKeysGm_, outValuesGm_, hitFlagsGm_, probeCountsGm_;
    AscendC::TQue<AscendC::TPosition::VECIN, 2> queryQueue_;
    AscendC::TQue<AscendC::TPosition::VECOUT, 2> valueQueue_, hitQueue_, probeQueue_;
    uint32_t startQuery_ = 0;
    uint32_t localQueryCount_ = 0;
};

// legacy ascendc_library 通过该标准入口提取 Host Stub 并生成 ACLRT 启动头文件。
extern "C" __global__ __aicore__ void open_addressing_lookup_kernel(
    GM_ADDR tableKeys, GM_ADDR tableValues, GM_ADDR states, GM_ADDR queryKeys,
    GM_ADDR outValues, GM_ADDR hitFlags, GM_ADDR probeCounts, GM_ADDR tiling)
{
    AscendC::TPipe pipe;
    KernelOpenAddressingLookup op(&pipe);
    op.Init(tableKeys, tableValues, states, queryKeys,
            outValues, hitFlags, probeCounts,
            (__gm__ OpenAddressingLookupTilingData *)tiling);
    op.Process();
}

## 7. 编写 C++ Host 与二进制工具

In [ ]:
%%writefile $WRITE_ROOT/op_host/data_utils.h
#pragma once
#include <cstdint>
#include <fstream>
#include <stdexcept>
#include <string>
#include <vector>

inline std::vector<int32_t> ReadInt32File(const std::string &path, size_t count) {
    std::ifstream input(path, std::ios::binary);
    if (!input) throw std::runtime_error("cannot open " + path);
    std::vector<int32_t> data(count);
    input.read(reinterpret_cast<char *>(data.data()), count * sizeof(int32_t));
    if (input.gcount() != static_cast<std::streamsize>(count * sizeof(int32_t)))
        throw std::runtime_error("unexpected file size: " + path);
    return data;
}

inline void WriteInt32File(const std::string &path, const int32_t *data, size_t count) {
    std::ofstream output(path, std::ios::binary);
    if (!output) throw std::runtime_error("cannot write " + path);
    output.write(reinterpret_cast<const char *>(data), count * sizeof(int32_t));
}

In [ ]:
%%writefile $WRITE_ROOT/op_host/open_addressing_lookup_main.cpp
#include <algorithm>
#include <chrono>
#include <cstdint>
#include <filesystem>
#include <iostream>
#include <stdexcept>
#include <unordered_set>
#include <vector>

#include "acl/acl.h"
#include "aclrtlaunch_open_addressing_lookup_kernel.h"
#include "data_utils.h"
#include "../op_kernel/open_addressing_lookup_tiling.h"

#define ACL_CHECK(call) do { \
    const aclError err = (call); \
    if (err != ACL_SUCCESS) throw std::runtime_error(#call " failed: " + std::to_string(err)); \
} while (0)

static uint32_t Hash32(int32_t key) {
    uint32_t value = static_cast<uint32_t>(key);
    value *= 0x9E3779B1U;
    value ^= value >> 16;
    return value;
}

static bool IsPowerOfTwo(uint32_t value) {
    return value > 0 && (value & (value - 1)) == 0;
}

static void ValidateTable(const std::vector<int32_t> &keys,
                          const std::vector<int32_t> &states,
                          uint32_t maxProbe) {
    const uint32_t tableSize = states.size();
    std::unordered_set<int32_t> seen;
    for (uint32_t slot = 0; slot < tableSize; ++slot) {
        const int32_t state = states[slot];
        if (state < 0 || state > 2) throw std::runtime_error("invalid state");
        if (state != 1) continue;
        if (!seen.insert(keys[slot]).second) throw std::runtime_error("duplicate FULL key");
        const uint32_t home = Hash32(keys[slot]) & (tableSize - 1);
        const uint32_t distance = (slot - home) & (tableSize - 1);
        if (distance >= maxProbe) throw std::runtime_error("FULL key outside maxProbe");
        for (uint32_t step = 0; step < distance; ++step) {
            if (states[(home + step) & (tableSize - 1)] == 0)
                throw std::runtime_error("EMPTY slot breaks a probe chain");
        }
    }
}

int main(int argc, char **argv) {
    if (argc != 4) {
        std::cerr << "usage: open_addressing_lookup M Q MAX_PROBE\n";
        return 2;
    }
    const uint32_t tableSize = std::stoul(argv[1]);
    const uint32_t queryCount = std::stoul(argv[2]);
    const uint32_t maxProbe = std::stoul(argv[3]);
    if (!IsPowerOfTwo(tableSize) || queryCount == 0 || maxProbe == 0 || maxProbe > tableSize)
        throw std::runtime_error("invalid M, Q or MAX_PROBE");

    auto keys = ReadInt32File("input/table_keys.bin", tableSize);
    auto values = ReadInt32File("input/table_values.bin", tableSize);
    auto states = ReadInt32File("input/states.bin", tableSize);
    auto queries = ReadInt32File("input/query_keys.bin", queryCount);
    ValidateTable(keys, states, maxProbe);

    ACL_CHECK(aclInit(nullptr));
    ACL_CHECK(aclrtSetDevice(0));
    aclrtStream stream = nullptr;
    ACL_CHECK(aclrtCreateStream(&stream));
    int64_t availableCores = 0;
    ACL_CHECK(aclrtGetDeviceInfo(0, ACL_DEV_ATTR_VECTOR_CORE_NUM, &availableCores));
    const uint32_t blockNum = std::min<uint32_t>(queryCount, static_cast<uint32_t>(availableCores));
    OpenAddressingLookupTilingData tiling{
        tableSize, queryCount, maxProbe, blockNum,
        (queryCount + blockNum - 1) / blockNum, 0};

    std::vector<void *> deviceInputs(4, nullptr), deviceOutputs(3, nullptr);
    void *tilingDevice = nullptr;
    const size_t tableBytes = tableSize * sizeof(int32_t);
    const size_t queryBytes = queryCount * sizeof(int32_t);
    const void *hostInputs[4] = {keys.data(), values.data(), states.data(), queries.data()};
    const size_t inputBytes[4] = {tableBytes, tableBytes, tableBytes, queryBytes};
    std::vector<int32_t> outValues(queryCount), hitFlags(queryCount), probeCounts(queryCount);
    int32_t *hostOutputs[3] = {outValues.data(), hitFlags.data(), probeCounts.data()};

    for (int i = 0; i < 4; ++i) {
        ACL_CHECK(aclrtMalloc(&deviceInputs[i], inputBytes[i], ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMemcpy(deviceInputs[i], inputBytes[i], hostInputs[i], inputBytes[i], ACL_MEMCPY_HOST_TO_DEVICE));
    }
    for (int i = 0; i < 3; ++i)
        ACL_CHECK(aclrtMalloc(&deviceOutputs[i], queryBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&tilingDevice, sizeof(tiling), ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemcpy(tilingDevice, sizeof(tiling), &tiling, sizeof(tiling), ACL_MEMCPY_HOST_TO_DEVICE));

    const auto begin = std::chrono::steady_clock::now();
    // 启动宏本身不作为 aclError 返回值使用；随后由 stream 同步检查执行结果。
    ACLRT_LAUNCH_KERNEL(open_addressing_lookup_kernel)(
        blockNum, stream,
        deviceInputs[0], deviceInputs[1], deviceInputs[2], deviceInputs[3],
        deviceOutputs[0], deviceOutputs[1], deviceOutputs[2], tilingDevice);
    ACL_CHECK(aclrtSynchronizeStream(stream));
    const auto end = std::chrono::steady_clock::now();
    const double kernelUs = std::chrono::duration<double, std::micro>(end - begin).count();

    for (int i = 0; i < 3; ++i)
        ACL_CHECK(aclrtMemcpy(hostOutputs[i], queryBytes, deviceOutputs[i], queryBytes, ACL_MEMCPY_DEVICE_TO_HOST));
    std::filesystem::create_directories("output");
    WriteInt32File("output/npu_values.bin", outValues.data(), queryCount);
    WriteInt32File("output/npu_hits.bin", hitFlags.data(), queryCount);
    WriteInt32File("output/npu_probes.bin", probeCounts.data(), queryCount);
    std::cout << "[PERF] kernel_us=" << kernelUs
              << " query_per_second=" << queryCount * 1e6 / kernelUs << "\n";

    aclrtFree(tilingDevice);
    for (void *ptr : deviceInputs) aclrtFree(ptr);
    for (void *ptr : deviceOutputs) aclrtFree(ptr);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}

## 8. 编写结果验证与非法输入测试

In [ ]:
%%writefile $WRITE_ROOT/scripts/verify_result.py
import json
from pathlib import Path
import numpy as np

meta = json.loads(Path("meta.json").read_text(encoding="utf-8"))
count = int(meta["query_count"])
pairs = [
    ("value", "output/golden_values.bin", "output/npu_values.bin"),
    ("hit", "output/golden_hits.bin", "output/npu_hits.bin"),
    ("probe", "output/golden_probes.bin", "output/npu_probes.bin"),
]
for label, golden_path, npu_path in pairs:
    golden = np.fromfile(golden_path, dtype=np.int32)
    actual = np.fromfile(npu_path, dtype=np.int32)
    if golden.size != count or actual.size != count:
        raise AssertionError(f"{label} output size mismatch")
    mismatch = np.flatnonzero(golden != actual)
    if mismatch.size:
        i = int(mismatch[0])
        raise AssertionError(f"{label}[{i}]: golden={golden[i]}, npu={actual[i]}")
print("PASSED: value/hit/probe outputs exactly match CPU Golden")

In [ ]:
%%writefile $WRITE_ROOT/scripts/validate_inputs.py
import numpy as np
from open_addressing_utils import LinearProbeTable, validate_arrays

table = LinearProbeTable(8)
table.insert(7, 70)
validate_arrays(table.keys, table.values, table.states, 8)

checks = 0
def must_fail(action):
    global checks
    try:
        action()
    except ValueError:
        checks += 1
        return
    raise AssertionError("invalid input was not rejected")

must_fail(lambda: validate_arrays(np.zeros(6, np.int32), np.zeros(6, np.int32), np.zeros(6, np.int32), 6))
bad_states = table.states.copy(); bad_states[0] = 9
must_fail(lambda: validate_arrays(table.keys, table.values, bad_states, 8))
must_fail(lambda: validate_arrays(table.keys, table.values, table.states, 0))
must_fail(lambda: validate_arrays(table.keys, table.values, table.states, 9))

# 构造实际距离为 2 的碰撞记录，再验证过小 MAX_PROBE 会被拒绝。
collision = LinearProbeTable(8)
from open_addressing_utils import keys_for_home
a, b = keys_for_home(3, 2, 8)
collision.insert(a, 1); collision.insert(b, 2)
must_fail(lambda: validate_arrays(collision.keys, collision.values, collision.states, 1))
broken = collision.states.copy(); broken[(collision.states == 1).nonzero()[0][0]] = 0
must_fail(lambda: validate_arrays(collision.keys, collision.values, broken, 8))
print(f"PASSED: {checks} invalid-input checks")

In [ ]:
subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts/validate_inputs.py")],
    cwd=WORK_DIR, check=True,
)

## 9. 配置 CMake 与一键运行脚本

In [ ]:
%%writefile $WRITE_ROOT/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(open_addressing_lookup LANGUAGES CXX)
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)
# CANN 9.0 legacy 的 merge_mix_obj.sh 要求 --build-type 后必须有非空值。
if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()
if(NOT DEFINED SOC_VERSION)
  set(SOC_VERSION ascend910b3)
endif()
if(NOT DEFINED ENV{ASCEND_HOME_PATH})
  message(FATAL_ERROR "ASCEND_HOME_PATH is not set")
endif()
set(ASCEND_CANN_PACKAGE_PATH "$ENV{ASCEND_HOME_PATH}" CACHE PATH "CANN path" FORCE)
set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "output path" FORCE)
set(ASCENDC_CMAKE_CANDIDATES
  "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
foreach(candidate IN LISTS ASCENDC_CMAKE_CANDIDATES)
  if(EXISTS "${candidate}")
    set(ASCENDC_CMAKE_FILE "${candidate}")
    break()
  endif()
endforeach()
if(NOT ASCENDC_CMAKE_FILE)
  message(FATAL_ERROR "Cannot find ascendc.cmake")
endif()
include("${ASCENDC_CMAKE_FILE}")
ascendc_library(open_addressing_lookup_kernels STATIC
  op_kernel/open_addressing_lookup_kernel.cpp)
ascendc_include_directories(open_addressing_lookup_kernels PRIVATE op_kernel)
ascendc_compile_definitions(open_addressing_lookup_kernels PRIVATE
  ASCENDC_DUMP=0 ASCENDC_DUMP_BBOX=0)
add_executable(open_addressing_lookup op_host/open_addressing_lookup_main.cpp)
target_include_directories(open_addressing_lookup PRIVATE
  op_host op_kernel
  ${ASCEND_CANN_PACKAGE_PATH}/include
  ${ASCEND_CANN_PACKAGE_PATH}/include/external
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
  ${CMAKE_INSTALL_PREFIX}/include/open_addressing_lookup_kernels
  ${CMAKE_BINARY_DIR}/out/include/open_addressing_lookup_kernels)
target_link_directories(open_addressing_lookup PRIVATE
  ${ASCEND_CANN_PACKAGE_PATH}/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64)
target_link_libraries(open_addressing_lookup PRIVATE
  open_addressing_lookup_kernels ascendcl runtime pthread dl)

In [ ]:
%%writefile $WRITE_ROOT/run.sh
#!/usr/bin/env bash
set -euo pipefail

# 完整回归只构建一次；--skip-build 可直接复用已有产物。
ROOT_DIR="$(cd "$(dirname "$0")" && pwd)"
SKIP_BUILD=0
if [[ "${1:-}" == "--skip-build" ]]; then SKIP_BUILD=1; fi
if [[ -z "${ASCEND_HOME_PATH:-}" || ! -f "${ASCEND_HOME_PATH}/set_env.sh" ]]; then
  echo "ASCEND_HOME_PATH/set_env.sh not found" >&2
  exit 2
fi
source "${ASCEND_HOME_PATH}/set_env.sh"
python3 "${ROOT_DIR}/scripts/validate_inputs.py"

if [[ ${SKIP_BUILD} -eq 0 ]]; then
  rm -rf "${ROOT_DIR}/build"
  mkdir -p "${ROOT_DIR}/build"
  cmake -S "${ROOT_DIR}" -B "${ROOT_DIR}/build" -DSOC_VERSION=ascend910b3 -DCMAKE_BUILD_TYPE=Release
  # 首次构建使用单线程，避免多个 legacy 子工程的教学日志相互交错。
  cmake --build "${ROOT_DIR}/build" --verbose -j1
fi
cd "${ROOT_DIR}/build"
for CASE in empty basic cluster tombstone tail130 load25 load50 load75 negative_zero; do
  echo "[CASE] ${CASE}"
  python3 "${ROOT_DIR}/scripts/gen_data.py" --case "${CASE}"
  read -r M Q P < <(python3 -c 'import json; m=json.load(open("meta.json")); print(m["table_size"],m["query_count"],m["max_probe"])')
  ./open_addressing_lookup "${M}" "${Q}" "${P}"
  python3 "${ROOT_DIR}/scripts/verify_result.py"
done
echo "ALL CASES PASSED"

In [ ]:
%%writefile $WRITE_ROOT/README.md
# OpenAddressingLookup 直调实验工程

该目录由 `09.02_open_addressing_lookup.ipynb` 生成。Host 构建只读开放寻址表，Ascend C Kernel 按连续 query Tile 执行有限步线性探测。

```bash
bash run.sh
bash run.sh --skip-build
```

输出包括 value、hit 和 probe count；`probe count` 是算法访问次数，不能直接替代 NPU 性能数据。

## 10. 编译工程

In [ ]:
if NPU_READY:
    build_dir = WORK_DIR / "build"
    if build_dir.exists():
        shutil.rmtree(build_dir)
    build_dir.mkdir(parents=True)
    command = (
        'source "$ASCEND_HOME_PATH/set_env.sh" && '
        # 单线程构建让 CANN 9.0 legacy 子工程日志按执行顺序显示。
        'cmake .. -DSOC_VERSION=ascend910b3 -DCMAKE_BUILD_TYPE=Release && '
        'cmake --build . --verbose -j1'
    )
    result = subprocess.run(
        ["bash", "-lc", command], cwd=build_dir,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    log_path = build_dir / "build.log"
    log_path.write_text(result.stdout, encoding="utf-8")
    # 失败时多显示一些上下文；完整输出同时保存在 build/build.log。
    print("\n".join(result.stdout.splitlines()[-120:]))
    if result.returncode:
        raise RuntimeError(f"编译失败，完整日志：{log_path}")
    print("编译完成：", build_dir / "open_addressing_lookup")
else:
    print("跳过 NPU 编译；请在 CANNLab 910B3 环境重新运行本单元。")

## 11. 运行单个案例

In [ ]:
CASE_NAME = "tombstone"  # 可改为 basic、cluster、tail130、load75 等。
if NPU_READY:
    build_dir = WORK_DIR / "build"
    subprocess.run(
        [sys.executable, str(WORK_DIR / "scripts/gen_data.py"), "--case", CASE_NAME],
        cwd=build_dir, check=True,
    )
    import json
    meta = json.loads((build_dir / "meta.json").read_text(encoding="utf-8"))
    command = (
        'source "$ASCEND_HOME_PATH/set_env.sh" && '
        f'./open_addressing_lookup {meta["table_size"]} {meta["query_count"]} {meta["max_probe"]}'
    )
    subprocess.run(["bash", "-lc", command], cwd=build_dir, check=True)
    subprocess.run(
        [sys.executable, str(WORK_DIR / "scripts/verify_result.py")],
        cwd=build_dir, check=True,
    )
else:
    print("跳过 NPU 单案例；CPU 数据生成与统计仍可运行。")

## 12. 执行完整回归

完整回归构建一次后依次运行 9 个合法案例。若已经完成第 10 节编译，可把命令改为 `bash run.sh --skip-build` 复用产物。运行期间会持续打印 `[CASE]`、性能行和验证结果。

In [ ]:
if NPU_READY:
    subprocess.run(["bash", "run.sh", "--skip-build"], cwd=WORK_DIR, check=True)
else:
    print("跳过 NPU 完整回归；请在 CANNLab 910B3 环境执行。")

## 13. 实验总结

本实验建立了从装填因子到探测长度，再到 NPU 不规则表访问的完整观察链路。降低装填因子会使用更多表空间，但通常能缩短平均和长尾探测；SoA 允许未命中路径跳过 value 读取；连续 query Tiling 则保证输入与输出能够规则分核和连续搬运。实际性能仍应结合当前数据分布、Kernel 耗时和吞吐测量判断。

## 课后思考

1. 为什么把 `TOMBSTONE` 当作 `EMPTY` 会造成错误未命中？
2. 装填因子升高时，为什么 P99 和最大探测长度通常比平均值增长更明显？
3. SoA 布局为什么能够减少未命中查询的无效 value 读取？
4. 为什么 `probe_counts` 不能直接等同于 NPU Kernel 耗时？

In [ ]:
!cat answer/09.02_open_addressing_lookup/answers.md